# Drive failure prediction from 30-day SMART windows — the whole Backblaze archive

Three steps over **every drive Backblaze has ever published telemetry for**:

| step | question | output |
|---|---|---|
| **1** | is any column a *proxy* for `failure` rather than a predictor of it? | a scored audit, a blocklist, and the surviving feature set $X$ |
| **2** | what does a 30-day window cover, and how many are there? | a window index over *every possible start offset*, and the timeline histograms |
| **3** | how well does a sequence model rank drives about to fail? | per-epoch logs, test metrics, confusion matrix, Precision@K |

**The data.** The Backblaze Drive Stats archive in full: three yearly zips (2013–2015)
and one per quarter from Q1 2016 on, fetched and reduced to parquet shards by
`scripts/build_backblaze_archive.py`. Every manufacturer, every model, every drive —
**no vendor filter and no cohort sampling**.

That last point is the important difference from this project's earlier runs. The
Toshiba export was a **matched cohort**: every drive that failed plus a sample of drives
that did not, so roughly half of it carried a failure. This is the **fleet**, at its real
prevalence — failures are a fraction of a percent of drive-days. Precision numbers from a
cohort are inflated by construction and do not transfer; these do.

**Every possible window.** A sample is one drive's 30 consecutive daily readings, and the
label is *does this drive fail within 30 days of the window's last day*. For evaluation
the window start is enumerated at **stride 1** — every offset from day 0 to
`length − 30` for every drive in the split. Nothing is skipped and nothing is sampled.

In [1]:
import gc, json, os, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "scripts")
import leakage_audit as la
import backblaze_window_pipeline as bw
import archive_window_pipeline as ap

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 9})
pd.set_option("display.width", 200, "display.max_columns", 40)

SHARD_DIR = os.environ.get("BACKBLAZE_SHARDS",
                           os.path.join("data", "backblaze_archive", "shards"))
STORE_DIR = os.environ.get("BACKBLAZE_STORE",
                           os.path.join("data", "backblaze_archive", "store"))
FIG_DIR   = "results/figures"
SEED      = 42
os.makedirs(FIG_DIR, exist_ok=True)
bw.set_seed(SEED)

shards = ap.shard_paths(SHARD_DIR)
print(f"torch {torch.__version__} | device "
      f"{'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"{len(shards)} archive shards: {os.path.basename(shards[0])[:-8]} "
      f"-> {os.path.basename(shards[-1])[:-8]}")

torch 2.14.0.dev20260708+cu132 | device cuda
45 archive shards: 2013 -> Q2_2026


---
## Step 1 — Per-drive telemetry and target-leakage analysis

A leaked column is one that is not *evidence about* the outcome but a *restatement of*
it — the "failing grade of 40" used to predict pass/fail. In drive telemetry that column
is almost never a SMART attribute. It hides in three other places, and the audit probes
all three:

1. **Column values.** Every telemetry column scored against the drive-day `failure` flag
   with a point-biserial correlation, a rank AUC, and the best single-threshold balanced
   accuracy. One column that separates the classes at AUC ≈ 1.0 on its own is a
   giveaway.

2. **Missingness.** A column can leak through *whether it is null* while its values stay
   innocuous — an operational pipeline that stops populating a field once a drive is
   pulled stamps the outcome onto the row. Scored as
   $P(\text{null} \mid \text{failure}) - P(\text{null} \mid \text{healthy})$.

3. **Record geometry.** The big one. Backblaze writes `failure = 1` on a drive's **last
   reporting day**, so anything derived from where a row sits relative to the end of that
   drive's record reconstructs the label exactly. These columns are not in the file — they
   are what a feature-engineering step would *create* — so the audit builds them itself
   and scores them as a warning.

**The audit runs on a sample, and the sample is drawn by *drive*, not by drive-day.**
Scoring 700M+ rows is neither necessary nor affordable, but thinning rows would break the
audit outright: every geometry probe is derived from a drive's own first and last day, and
a drive whose rows have been thinned has neither. Whole drives keep the geometry exact and
keep the fleet failure base rate intact.

In [2]:
t0 = time.time()
cfg0 = ap.ArchiveConfig(shard_dir=SHARD_DIR, store_dir=STORE_DIR, random_state=SEED)
census = ap.build_drive_census(cfg0)

print(f"\n{len(census):,} drives in the archive | "
      f"{int(census.n_obs.sum()):,} observed drive-days | "
      f"{int(census.has_failure.sum()):,} drives with a failure event")
print(f"{pd.Timestamp('1970-01-01') + pd.Timedelta(days=int(census.start_day.min())):%Y-%m-%d}"
      f" -> "
      f"{pd.Timestamp('1970-01-01') + pd.Timedelta(days=int(census.end_day.max())):%Y-%m-%d}"
      f"  ({time.time()-t0:.0f}s)")

  census cached: 512,600 drives

512,600 drives in the archive | 744,525,683 observed drive-days | 33,566 drives with a failure event
2013-04-10 -> 2026-06-30  (0s)


In [3]:
audit_df = ap.sample_audit_frame(cfg0, census, n_drives=15_000)

  [ 1/45] 2013.parquet       152,962 rows kept


  [ 2/45] 2014.parquet       380,844 rows kept


  [ 3/45] 2015.parquet       525,518 rows kept


  [ 4/45] Q1_2016.parquet    164,588 rows kept


  [ 5/45] Q2_2016.parquet    178,162 rows kept


...

  [42/45] Q3_2025.parquet    868,235 rows kept


  [43/45] Q4_2025.parquet    901,168 rows kept


  [44/45] Q1_2026.parquet    892,179 rows kept


  [45/45] Q2_2026.parquet    929,828 rows kept


  audit frame: 21,654,640 drive-days over 15,000 drives, 991 failure events (base rate 0.00458%)


In [4]:
report = la.audit_frame(audit_df)
report.print_report()

[audit] scoring 23 telemetry columns, 6 identifier/topology columns, and 6 derived record-geometry probes over 21,654,640 drive-days ...


STEP 1 -- TARGET LEAKAGE AND PROXY AUDIT
  drive-days ........... 21,654,640
  distinct drives ...... 15,000
  failure events ....... 991 (base rate 0.0046%)

  thresholds: severe if rank-AUC >= 0.95, or 1-threshold balanced acc >= 0.90,
              or |r| >= 0.50, or missingness gap >= 0.25, or level lift >= 20x

--------------------------------------------------------------------------------------------
A. TELEMETRY COLUMNS vs failure -- top 12 by rank AUC
--------------------------------------------------------------------------------------------
       column coverage  n_unique   corr auc_directional balanced_acc verdict
smart_187_raw   0.4744       852 0.0306          0.8283       0.8231 suspect
smart_197_raw   0.9817      1447 0.0467          0.7509       0.7498   clean
  smart_5_raw   0.9955      7482 0.0224          0.7459       0.7421   clean
smart_198_raw   0.9863       877 0.0460          0.6893       0.6886   clean
  smart_4_raw   0.9848      1115 0.0004          0.6330  

### Reading the audit

The same three findings the cohort run produced, now measured on the fleet:

**No raw SMART attribute is a proxy.** The strongest reach a rank AUC in the 0.7–0.8
range — high enough to be genuinely useful, nowhere near the ≥ 0.95 that would mean a
column *is* the label. That is the result we want from section A: the telemetry is
signal, not an echo.

**Fleet-topology columns leak, mostly through absence.** `pod_slot_num`, `vault_id`,
`cluster_id` and `datacenter` describe where a disk sat, not how it was behaving, and
they stop being populated once a disk is pulled. They are carried through ingest
*precisely so the audit can score them* and are then blocked.

**Record geometry is the severe leak.** `days_to_last_record` and
`still_reporting_at_export_end` reconstruct the label directly, because `failure = 1` is
written on a drive's last reporting day. Nothing derived from a record boundary may enter
$X$ — and unlike the cohort run, here `still_reporting_at_export_end` is *also* a
statement about the fleet's growth over thirteen years, which makes it more tempting and
no less fatal.

**Coverage does the rest of the work.** Backblaze has added and dropped SMART columns
repeatedly since 2013 — the 2013 files populate only five attributes — so an attribute
reported by a minority of the archive is dropped on coverage before leakage is even
considered.

In [5]:
features = la.select_feature_columns(audit_df, report)

print(f"{len(features)} attributes survive coverage >= {report.cfg.min_coverage:.0%} "
      f"and the blocklist:\n")
tbl = report.columns.set_index("column")
for c in features:
    r = tbl.loc[c]
    print(f"  {c:<18} coverage {100*r.coverage:6.2f}%  "
          f"{int(r.n_unique):>8,} distinct  rank AUC {r.auc_directional:.4f}")

# The guard: raises if any blocklisted name, or anything derived from one, reached X.
report.assert_clean(features)

15 attributes survive coverage >= 50% and the blocklist:

  smart_197_raw      coverage  98.17%     1,447 distinct  rank AUC 0.7509
  smart_5_raw        coverage  99.55%     7,482 distinct  rank AUC 0.7459
  smart_198_raw      coverage  98.63%       877 distinct  rank AUC 0.6893
  smart_4_raw        coverage  98.48%     1,115 distinct  rank AUC 0.6330
  smart_12_raw       coverage  99.00%       345 distinct  rank AUC 0.6217
  smart_240_raw      coverage  67.28%   152,594 distinct  rank AUC 0.6095
  smart_193_raw      coverage  98.07%   186,068 distinct  rank AUC 0.6082
  smart_9_raw        coverage  99.87%    82,648 distinct  rank AUC 0.5883
  smart_1_raw        coverage  99.83%  7,986,909 distinct  rank AUC 0.5788
  smart_7_raw        coverage  98.48%  8,976,066 distinct  rank AUC 0.5750
  smart_3_raw        coverage  98.48%     3,826 distinct  rank AUC 0.5490
  smart_194_raw      coverage  99.87%        58 distinct  rank AUC 0.5222
  smart_199_raw      coverage  98.67%       968 dist

In [6]:
la.plot_audit(report, save_path=f"{FIG_DIR}/step1_leakage_audit.png")

  saved results/figures/step1_leakage_audit.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\leakage_audit.py:618: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1650x990 with 4 Axes>

In [7]:
del audit_df; gc.collect()

40

---
## Step 2 — Every possible 30-day window

A sample is one drive's **30 consecutive daily readings**, shape `(30, n_channels)`, and
the label is *does this drive fail within 30 days of the window's last day*. The 30-day
horizon is what makes the task a maintenance question rather than a detection one: a
window that ends 30 days before the disk dies is a **useful** warning, and under the
strict "failure inside the window" rule it would be labelled negative.

**Where a window may start.** Each drive is laid on a gap-free daily calendar — a drive
seen on the 1st and the 5th gets rows for the 2nd, 3rd and 4th, forward-filled from the
last real reading and flagged `observed = 0` so the model can discount them. A window is
a contiguous slice of that calendar. For validation and test the start is enumerated at
**stride 1**: every offset in `[0, length − 30]`, for every drive. Drives whose record is
shorter than 30 days are dropped rather than padded, so every window really is 30
continuous days.

**Why this needs a different store.** At archive scale the feature matrix is ~140 GB
against 103 GB of RAM, so it lives on disk as a memory-mapped array built by an external
sort — the shards arrive time-major and the store needs drive-major. The window index is
never materialised either: per-drive window *counts* plus a `searchsorted` turn a global
window id into `(drive, start)` arithmetically, which is the difference between ~5 MB of
index and the ~6 GB an explicit list would cost. The batch is the unit of work; nothing
in the hot path runs per window in Python.

**The leakage guard, again.** The failure flag lives in `store.failure`, a separate array
from `store.values`. There is no code path by which a model reads the label out of $X$ —
the window tensor contains SMART channels, their 1- and 7-day deltas, and the observed
mask, and nothing else.

In [8]:
cfg = ap.ArchiveConfig(
    shard_dir         = SHARD_DIR,
    store_dir         = STORE_DIR,
    window_days       = 30,    # a sample is 30 consecutive days
    horizon_days      = 30,    # positive if the drive fails within 30 days of window end
    stride_days       = 1,     # EVERY possible window start is enumerated for eval
    min_days          = 30,    # shorter drives are dropped, not left-padded
    delta_lags        = (1, 7),
    add_observed_mask = True,
    feature_columns   = tuple(features),
    log1p_columns     = tuple(c for c in features
                              if c not in ("smart_194_raw", "smart_190_raw")),
    n_buckets         = 64,    # external-sort width
    samples_per_drive = 4,     # random training windows per drive per epoch
    train_positive_ratio = 0.25,
    batch_size        = 4096,
    random_state      = SEED,
)

store = ap.build_archive_store(cfg)

STEP A -- DRIVE CENSUS
  census cached: 512,600 drives

STEP B -- DENSE LAYOUT


  508,653 drives kept, 3,947 dropped (shorter than 30 days)
  752,397,855 dense drive-days (744,456,164 observed, 7,941,691 forward-filled)
  matrix 752,397,855 x 46 float32 = 138.4 GB memory-mapped

STEP C -- EXTERNAL SORT INTO 64 BUCKETS
  buckets cached

STEP D -- DENSIFY
  dense matrix cached

ArchiveSeriesStore: 508,653 drives, 752,397,855 dense drive-days, 46 channels (138 GB memory-mapped), 32,707 drives with a failure event


In [9]:
# Stride 1 everywhere -- training, validation and test all see every possible window.
# An earlier version ran per-epoch validation at a coarser stride because the gather was
# the bottleneck; reading each batch as one contiguous slab instead of 4,096 scattered
# window lookups removed the need for that compromise.
bundle = ap.build_archive_bundle(cfg, store=store, val_stride=1)


grouped split by drive id -- no drive appears in two splits:
  train  345,884 drives  (22,241 carry a failure, 6.43%)
  val     61,039 drives  (3,925 carry a failure, 6.43%)
  test   101,730 drives  (6,541 carry a failure, 6.43%)
  scaler cached -- matrix already standardised in place

Window datasets (window_days=30, horizon_days=30, stride_days=1):
  train | random spd 4   |    1,383,536 windows | 345,884 drives | positive rate  1.3208% (measured, 22,241 failing drives)


  val   | enumerate stride 1  |   88,632,676 windows |  61,039 drives | positive rate  0.1381% (122,444 positive windows)


  test  | enumerate stride 1  |  147,297,386 windows | 101,730 drives | positive rate  0.1382% (203,507 positive windows)

Raw pos_weight from the training positive rate: 74.7


### How many windows is "every possible window"?

The count below is exact, not an estimate: it is
$\sum_{\text{drives}} (\text{length} - 30 + 1)$ over each split. The evaluation really
does score every one of them — the batching exists so that it can, not so that it can
skip any.

In [10]:
rows = []
for name in ("train", "val", "test"):
    ds = bundle.datasets[name]
    idx = bundle.splits[name]
    n_possible = int((store.lengths[idx] - cfg.window_days + 1).sum())
    rows.append({
        "split": name,
        "drives": len(idx),
        "drives with a failure": int(store.drive_has_failure[idx].sum()),
        "dense drive-days": int(store.lengths[idx].sum()),
        "every possible 30-day window": n_possible,
        "windows this split uses": int(ds.n_windows if hasattr(ds, "n_windows")
                                       else ds.n_samples),
        "batches": len(ds),
    })
coverage = pd.DataFrame(rows).set_index("split")
print(coverage.to_string(formatters={c: "{:,}".format for c in coverage.columns
                                     if coverage[c].dtype.kind in "iu"}))

# The guarantee, asserted rather than asserted-in-prose: at stride 1 the number of
# windows the test split enumerates is exactly the number that exist.
full_test = bundle.datasets["test"]
expected = int((store.lengths[bundle.splits["test"]] - cfg.window_days + 1).sum())
assert cfg.stride_days == 1 and full_test.n_windows == expected, \
    (cfg.stride_days, full_test.n_windows, expected)
print(f"\ntest split enumerates every one of {full_test.n_windows:,} possible windows "
      f"in {len(full_test):,} batches of {cfg.batch_size}")

       drives drives with a failure dense drive-days every possible 30-day window windows this split uses batches
split                                                                                                            
train 345,884                22,241      511,747,492                  501,716,856               1,383,536     338
val    61,039                 3,925       90,402,807                   88,632,676              88,632,676  21,639
test  101,730                 6,541      150,247,556                  147,297,386             147,297,386  35,962

test split enumerates every one of 147,297,386 possible windows in 35,962 batches of 4096


### The window timeline

The training sampler draws a start uniformly from `[0, length − 30]`, seeded by
`(seed, epoch, batch)` rather than from global RNG state, so a new epoch re-rolls every
drive's window and the same call reproduces the same windows wherever it runs. The plots
below show where those windows actually land.

In [11]:
timeline = ap.sample_window_timeline(store, bundle.splits, n_per_drive=2,
                                     max_drives=40_000, seed=SEED)
ap.describe_window_timeline(timeline, cfg)
timeline.head(8)

STEP 2 -- RANDOM 30-DAY CONTINUOUS WINDOW SAMPLING
  windows drawn ........ 240,000 over 120,000 drives
  positive windows ..... 1,206 (0.5025%)
  start dates .......... 2013-04-10 .. 2026-06-01  (4,801 distinct start days)
  end dates ............ 2013-05-09 .. 2026-06-30
  calendar span ........ 30..30 days -- every window is exactly 30 continuous days
  genuine readings ..... mean 29.7/30 days per window (0.3 forward-filled)
  quarter-aligned ...... 1.010% of starts land on one of the 52 quarter boundaries (uniform expectation 1.083%)
  month-aligned ........ 3.393% (uniform expectation 3.291%)
                         starts are NOT snapped to a calendar grid
  distinct start days .. 4,801 of 4,801 calendar days in range
  weekday spread ....... Fri 14.3%, Mon 14.4%, Sat 14.3%, Sun 14.4%, Thu 14.3%, Tue 14.2%, Wed 14.1%


,split,drive_index,start_date,end_date,start_offset_days,start_fraction_of_life,drive_record_days,n_real_days,n_filled_days,n_padded_days,label
0,train,0,2026-05-26,2026-06-24,1438,0.995845,1474,30,0,0,0
1,train,0,2022-08-16,2022-09-14,59,0.040859,1474,30,0,0,0
2,train,5,2022-09-29,2022-10-28,434,0.244507,1805,30,0,0,0
3,train,5,2024-06-17,2024-07-16,1061,0.597746,1805,28,2,0,0
4,train,6,2023-10-07,2023-11-05,687,0.415106,1685,30,0,0,0
5,train,6,2024-10-09,2024-11-07,1055,0.637462,1685,30,0,0,0
6,train,19,2024-12-05,2025-01-03,771,0.586758,1344,30,0,0,0
7,train,19,2025-08-27,2025-09-25,1036,0.788432,1344,30,0,0,0


In [12]:
ap.plot_window_timeline(timeline, cfg, coverage=coverage,
                        save_path=f"{FIG_DIR}/step2_window_timeline.png")

  saved results/figures/step2_window_timeline.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\archive_window_pipeline.py:1586: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1980x880 with 6 Axes>

### Reading the timeline plots

**Top left — start dates.** The dotted verticals are calendar year boundaries. If the
windows were quarter- or month-aligned the histogram would collapse onto those lines;
instead it is broad across the whole thirteen-year range, with starts on every day of the
week in roughly equal proportion. It rises over time because the fleet grew — there are
simply more drives reporting in 2025 than in 2014.

**Top right — end dates**, with positive windows overlaid. Positives cluster where
failures actually happened rather than spreading evenly, which is a property of the data,
not of the sampler.

**Bottom left — start position along each drive's own life**, normalised to $[0, 1]$.
Flat means the sampler is not biased toward the beginning or the end of a drive's record
— important, because "near the end of the record" is exactly the leaked signal Step 1
blocked, and a sampler that favoured late windows would smuggle it back in through the
sampling distribution.

**Bottom centre — genuine readings per window.** Most windows are 30 real daily readings;
the tail is drives with reporting gaps, forward-filled and marked in the `observed`
channel.

---
## Step 3 — Training with per-epoch logging

A 1D-CNN over the sequence: dilated temporal blocks (a TCN, dilations 1/2/4/8) so the
receptive field spans the full 30 days without a recurrence, then global pooling to one
logit per window. `(batch, 30, channels) → (batch, 1)`.

**On the class imbalance.** This is the fleet, not a cohort, so the imbalance is severe —
a fraction of a percent of windows are positive. The single biggest mistake in this
project's earlier models was correcting that *twice* — a balanced sampler **and** a large
`pos_weight`, which distorts the gradient without improving the ranking every headline
metric here measures. So exactly one correction is applied: the training sampler forces
25% of the windows drawn from failing drives to be positive ones, and the loss is plain
BCE.

**Early stopping on validation PR-AUC**, not on loss or accuracy. At this positive rate
accuracy is meaningless — a model that predicts "healthy" every time scores over 99.6% —
and the loss moves with the imbalance correction rather than with the ranking that
actually gets used.

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tcfg = bw.TrainConfig(
    arch="cnn", hidden_dim=64, dropout=0.2,
    batch_size=cfg.batch_size, lr=1e-3, max_epochs=12, patience=3,
    loss="bce",        # one imbalance correction only -- the sampler does the other job
    sampler="none",
    # num_workers=0 on purpose. A batch is 4096 x 30 x 46 float32 = 23 MB, and pushing
    # that through worker IPC costs several times what the gather itself costs -- 8
    # workers measured 7x SLOWER than none. The slab read is already fast enough that
    # the GPU, not the disk, is the limit.
    num_workers=0, seed=SEED,
)

loaders = ap.build_loaders(bundle, tcfg, device)
model = bw.build_model(bundle.num_features, cfg.window_days, tcfg).to(device)

with torch.no_grad():
    probe = torch.zeros(2, cfg.window_days, bundle.num_features, device=device)
    print(f"input (2, {cfg.window_days}, {bundle.num_features}) -> "
          f"{tuple(model(probe).shape)}  (one logit per window)")
print(f"{bw.count_parameters(model):,} trainable parameters on {device}")

input (2, 30, 46) -> (2, 1)  (one logit per window)
136,769 trainable parameters on cuda


In [14]:
model, history, best_val = ap.train_archive_model(model, loaders, bundle, tcfg, device)
print(f"\nBest validation PR-AUC {best_val:.4f} at epoch "
      f"{1 + int(np.nanargmax(history['val_pr_auc']))}")

Loss: BCEWithLogitsLoss(pos_weight=1.0)  [requested]


TRAINING -- CNN, 136,769 parameters, up to 12 epochs, early stop after 3 without a val PR-AUC gain
  338 training batches/epoch, 21,639 validation batches/epoch
epoch | train loss |  val loss | val PR-AUC | val ROC-AUC | val F1@.5 |       lr |     sec
------+------------+-----------+------------+-------------+-----------+----------+--------


      epoch 1 training  19.8%  loss 0.12042


      epoch 1 training  39.6%  loss 0.09472


      epoch 1 training  59.5%  loss 0.08543


...

      epoch 10 training  59.5%  loss 0.06208


      epoch 10 training  79.3%  loss 0.06217


      epoch 10 training  99.1%  loss 0.06210


   10 |    0.06209 |   0.02146 |     0.1195 |      0.8704 |    0.1968 | 1.00e-03 |  2105.1


early stopping at epoch 10 (best val PR-AUC 0.1237)

Best validation PR-AUC 0.1237 at epoch 7


In [15]:
bw.plot_training_history(history, save_path=f"{FIG_DIR}/step3_training_history.png")

  saved results/figures/step3_training_history.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\backblaze_window_pipeline.py:2301: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 1650x462 with 3 Axes>

### Final evaluation

The decision threshold is the only quantity carried out of validation, and it is fixed
**before** the test split is touched. Everything below — PR-AUC, ROC-AUC, precision,
recall, F1, the confusion matrix and Precision@K — comes from one pass at that frozen
threshold, over **every possible 30-day window** in each split.

Precision@K is reported two ways. **Ranking windows** is the raw metric; **ranking
drives** collapses each drive to its highest-scoring window first, and is the form that
matches the decision an operator actually makes — you pull *K disks*, not K windows, and
one bad drive contributing hundreds of overlapping windows should not fill the top of the
list on its own.

In [16]:
results = ap.evaluate_archive_final(model, loaders, bundle, tcfg, device,
                                    pk_ks=(10, 25, 50, 100, 250, 1000))
results["history"] = history

FINAL VALIDATION PASS -- every window at stride 1


    val: 1/21,639 batches (  0.0%)  15,044 windows/s


    val: 1,082/21,639 batches (  5.0%)  31,502 windows/s


    val: 2,163/21,639 batches ( 10.0%)  32,113 windows/s


    val: 3,244/21,639 batches ( 15.0%)  32,097 windows/s


...

    test: 35,961/35,962 batches (100.0%)  65,568 windows/s


TEST @ threshold 0.5
  windows evaluated .... 147,297,386  (203,507 positive, base rate 0.138%)
  PR-AUC ............... 0.1230   (a random ranker scores 0.0014)
  ROC-AUC .............. 0.8609
  threshold ............ 0.5000
  precision ............ 0.1424
  recall ............... 0.3152
  F1 ................... 0.1962
  confusion ............ TP 64136  FP 386240  FN 139371  TN 146707639
TEST @ tuned threshold 0.639360
  windows evaluated .... 147,297,386  (203,507 positive, base rate 0.138%)
  PR-AUC ............... 0.1230   (a random ranker scores 0.0014)
  ROC-AUC .............. 0.8609
  threshold ............ 0.6394
  precision ............ 0.2098
  recall ............... 0.2196
  F1 ................... 0.2146
  confusion ............ TP 44683  FP 168332  FN 158824  TN 146925547
TEST CONFUSION MATRIX @ 0.639360   (threshold 0.6394)
                       predicted 0    predicted 1
    actual 0      146,925,547        168,332   <- 168,332 false alarms
    actual 1          158,824 

TEST PRECISION@K -- ranking windows
  147,297,386 items ranked, 203,507 positive (base rate 0.138%)
       K |   hits |  precision |    best |  recall@K |     lift
  -------+--------+------------+---------+-----------+---------
      10 |      9 |     0.9000 |  1.0000 |    0.0000 |   651.4x
      25 |     22 |     0.8800 |  1.0000 |    0.0001 |   636.9x
      50 |     45 |     0.9000 |  1.0000 |    0.0002 |   651.4x
     100 |     91 |     0.9100 |  1.0000 |    0.0004 |   658.7x
     250 |    208 |     0.8320 |  1.0000 |    0.0010 |   602.2x
    1000 |    782 |     0.7820 |  1.0000 |    0.0038 |   566.0x


TEST PRECISION@K -- ranking DRIVES (each drive scored by its worst window)
  101,730 items ranked, 6,541 positive (base rate 6.430%)
       K |   hits |  precision |    best |  recall@K |     lift
  -------+--------+------------+---------+-----------+---------
      10 |      9 |     0.9000 |  1.0000 |    0.0014 |    14.0x
      25 |     24 |     0.9600 |  1.0000 |    0.0037 |    14.9x
      50 |     48 |     0.9600 |  1.0000 |    0.0073 |    14.9x
     100 |     91 |     0.9100 |  1.0000 |    0.0139 |    14.2x
     250 |    228 |     0.9120 |  1.0000 |    0.0349 |    14.2x
    1000 |    857 |     0.8570 |  1.0000 |    0.1310 |    13.3x


SUMMARY
  val   @tuned           PR-AUC 0.1237 | ROC-AUC 0.8670 | P 0.2115 | R 0.2152 | F1 0.2133 | thr 0.6394 | TP 26346 FP 98195 FN 96098 TN 88412037
  test  @0.50            PR-AUC 0.1230 | ROC-AUC 0.8609 | P 0.1424 | R 0.3152 | F1 0.1962 | thr 0.5000 | TP 64136 FP 386240 FN 139371 TN 146707639
  test  @tuned           PR-AUC 0.1230 | ROC-AUC 0.8609 | P 0.2098 | R 0.2196 | F1 0.2146 | thr 0.6394 | TP 44683 FP 168332 FN 158824 TN 146925547


In [17]:
ap.plot_evaluation(results, ks=(10, 25, 50, 100, 250, 1000),
                   save_path=f"{FIG_DIR}/step3_evaluation.png")

  curves drawn from 4,000,000 of 147,297,386 test windows (uniform subsample; all printed metrics use the full pass)


  saved results/figures/step3_evaluation.png


C:\Users\tempuser2\Desktop\XGBOOST_2.0\XGBOOST\scripts\backblaze_window_pipeline.py:2376: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<Figure size 2090x462 with 4 Axes>

In [18]:
print("Highest-risk drives in the test split (each scored by its worst window):\n")
print(results["top_drives"].head(20).to_string(index=False))

Highest-risk drives in the test split (each scored by its worst window):

 rank serial_number     risk  actually_failed
    1      S1F03TCS 0.999997                1
    2      W1F09AKF 0.999982                1
    3      S1F0AAQA 0.999892                0
    4      ZLW18S0J 0.999892                1
    5      Z305GVK2 0.999892                1
    6      S1F0CTVX 0.999890                1
    7      ZHZ39HP6 0.999882                1
    8      ZHZ3PJXJ 0.999875                1
    9      ZLW0H2T5 0.999871                1
   10      ZHZ5GPWY 0.999862                1
   11      ZL2L803G 0.999844                1
   12      W1F09X4B 0.999844                1
   13      Z302BV6W 0.999842                1
   14      ZA12DPQ7 0.999818                1
   15      W1F0A67D 0.999818                1
   16      Z305D63V 0.999804                1
   17      S1F04DF3 0.999792                1
   18      Z302A0MB 0.999788                1
   19      S1F030XE 0.999770                1
   20 

In [19]:
os.makedirs("models", exist_ok=True); os.makedirs("results", exist_ok=True)
ap.save_artifacts(model, bundle, tcfg, results, prefix="models/backblaze_archive30")

pd.DataFrame(history).to_csv("results/archive30_training_history.csv", index=False)
report.columns.to_csv("results/archive30_column_association.csv", index=False)
pd.concat([report.geometry, report.categoricals], ignore_index=True).to_csv(
    "results/archive30_leakage_probes.csv", index=False)
timeline.to_csv("results/archive30_sampled_windows.csv", index=False)
coverage.to_csv("results/archive30_window_coverage.csv")
results["top_drives"].to_csv("results/archive30_top_drives.csv", index=False)
pd.concat([results["precision_at_k_windows"].assign(ranked="windows"),
           results["precision_at_k_drives"].assign(ranked="drives")],
          ignore_index=True).to_csv("results/archive30_precision_at_k.csv", index=False)

with open("results/archive30_summary.json", "w", encoding="utf-8") as fh:
    json.dump({
        "features": list(features),
        "blocklist": report.blocklist,
        "n_drives": int(store.n_drives),
        "n_drive_days": int(store.n_rows),
        "n_val_windows": results["n_val_windows"],
        "n_test_windows": results["n_test_windows"],
        "best_threshold": results["best_threshold"],
        "val_at_tuned": results["val"],
        "test_at_tuned": results["test_at_tuned"],
        "test_at_half": results["test_at_half"],
    }, fh, indent=2)
print("wrote results/archive30_*.csv and results/archive30_summary.json")

Saved models/backblaze_archive30_cnn.pth and models/backblaze_archive30_preprocessing_config.json


wrote results/archive30_*.csv and results/archive30_summary.json


---
## What these numbers do and do not say

**They rank, and now the base rate is real too.** PR-AUC and Precision@K score the
ordering of drives by risk, and that ordering is what a maintenance queue consumes. Unlike
this project's earlier cohort runs, the prevalence here is the fleet's own — a fraction of
a percent of drive-days — so the precision figures are not inflated by the sampling and do
not need to be discounted before they are read.

**Precision is low in absolute terms, and that is the honest answer.** At a base rate
below one percent, a precision of even a few percent is a large multiple of chance; the
lift column in the Precision@K table is the number to read, not the raw precision. The
drive-level table is the one that matches an operator's decision.

**The evaluation windows overlap.** At stride 1 a drive contributes one window per day of
its record and consecutive ones share 29 of their 30 days, so window-level counts are not
independent samples. That is the cost of covering every possible window, and it is exactly
why the drive-level Precision@K is reported alongside: it is the read that does not
double-count.

**Thirteen years is not one fleet.** The archive spans drive models, capacities and
datacenters that never coexisted, and the SMART attributes populated in 2013 are not the
ones populated in 2026. The coverage filter in Step 1 keeps only attributes reported
across most of the archive, but a model trained over the whole span is answering a
different question from one trained on the current fleet.

**What Step 1 bought.** Every record-geometry feature the audit blocked would have pushed
these numbers higher, and every point of it would have been the archive's construction
scoring itself. The metrics above are lower than a leaky pipeline's and are the ones that
would survive contact with a real fleet.